In [10]:
using SumOfSquares
using DynamicPolynomials
using JuMP
using Dates
using Random, Combinatorics
using SCS
using DataFrames
using CSV

Helper functions

In [11]:
function multilinear_monomials(x, d, lowest_deg=1)
    n = length(x)
    monoms = []
    push!(monoms, 1)
    for k in lowest_deg:d
        for C in combinations(1:n, k)
            push!(monoms, prod(x[i] for i in C))
        end
    end
    return monoms
end

function multilinear(coeffs, x)
    func = 0
    for (C, coeff) in coeffs
        if C == (0,)
            func += coeff
        else
            func += coeff * prod(x[i] for i in C)
        end
    end
    return func
end

# Define a function to get the second derivative
function get_second_derivative(f, variables, i, j)
    return differentiate(differentiate(f, variables[i]), variables[j])
end


function build_multilin_var(model, monom, name)
    # Create a dictionary to store the coefficients w_C
    w = @variable(model, [monom], base_name=name)
    # Build the polynomial F
    F = sum(w[m] * m for m in monom)
    return F, w
end

# Define the main function to add SOS constraints
function f_sos_submod(model, x, f, t)
    n = length(x)
    for i in 1:n
        for j in i+1:n
            # Define the second derivative
            d2F = get_second_derivative(f, x, i, j)
            S = algebraicset([xi^2 - xi for xi in x if (xi != x[i])&&(xi != x[j])])
            @constraint(model, -d2F >= 0, domain = S, maxdegree = 2*t)
        end
    end
end

num_multilinear_monomials(n, d) = 1 + sum(binomial(n, k) for k in 1:d)

num_multilinear_monomials (generic function with 1 method)

2. Optimize over tsos functions

In [12]:
function t_sos_opt(c, n, d, t)
    # min cT hat F st F t-sos-submod, ||hat F||^2_2 <= 1, hat F is vector of coeffs of F
    model = SOSModel(SCS.Optimizer)
    
    @polyvar x[1:n]
    monoms = multilinear_monomials(x, d)

    # Build multilinears
    f, wf = build_multilin_var(model, monoms, "f")

    # Add constraint || hat F ||^2_2 <= 1
    @constraint(model, sum(wf[m]^2 for m in monoms) <= 1)
    
    # sos-submodular constraints
    f_sos_submod(model, x, f, t)

    # Set the least squares objective: sum_i (y_i - f(x_i))^2
    obj = sum(c[i]*wf[monoms[i]] for i in 1:length(monoms))
    @objective(model, Min, obj)
    
    # Optimze
    optimize!(model)
    solve_time = try
        MOI.get(backend(model), MOI.SolveTimeSec())
    catch
        NaN
    end
    term_status = termination_status(model)
    prim_status = primal_status(model)
    println("Termination status: ", term_status)
    return (
        solve_time = solve_time,
        termination_status = string(term_status),
        primal_status = string(prim_status),
    )
end

t_sos_opt (generic function with 1 method)

Example

In [13]:
seed = 0
n = 10
d = 4
t = 1

Random.seed!(seed)

num_coeffs = num_multilinear_monomials(n, d)
c = randn(num_coeffs)

run_result = t_sos_opt(c, n, d, t)

------------------------------------------------------------------
	       SCS v3.2.11 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 3356, constraints m: 5968
cones: 	  z: primal zero / dual free vars: 2610
	  q: soc vars: 388, qsize: 1
	  s: psd vars: 2970, ssize: 45
settings: eps_abs: 1.0e-04, eps_rel: 1.0e-04, eps_infeas: 1.0e-07
	  alpha: 1.50, scale: 1.00e-01, adaptive_scale: 1
	  max_iters: 100000, normalize: 1, rho_x: 1.00e-06
	  acceleration_lookback: 10, acceleration_interval: 10
	  compiled with openmp parallelization enabled
lin-sys:  sparse-direct-amd-qdldl
	  nnz(A): 7991, nnz(P): 0
------------------------------------------------------------------
 iter | pri res | dua res |   gap   |   obj   |  scale  | time (s)
------------------------------------------------------------------
     0| 9.86e+01  2.33e+00  2.60e+03 -1.31e+03  1.00e-01  4.14e-02 
   250| 

(solve_time = 0.185662083, termination_status = "OPTIMAL", primal_status = "FEASIBLE_POINT")